# YouTube API -> Snowflake: initial load

Stage 1 of the pipeline: a one-off bootstrap load of every video for every
channel in `youtuber_analysis/config/channels.yaml` into a single Snowflake
table, `RAW.YOUTUBE_VIDEOS`. Dagster takes over keeping it current later;
nothing here depends on Dagster.

Uses the quota-cheap endpoint chain (see `youtuber_analysis/resources/youtube_api.py`):

| Call | Cost |
|------|------|
| `channels.list` | 1 unit per channel |
| `playlistItems.list` | 1 unit per 50 videos |
| `videos.list` | 1 unit per 50 videos |

**Table grain:** one row per video per `SNAPSHOT_DATE`. Re-running on the same
day replaces that day's rows for the channels being loaded, so it is safe to
re-run. Running on a new day appends a new snapshot.

Reads credentials from the repo-root `.env` (`YOUTUBE_API_KEY`, `SNOWFLAKE_*`).

In [28]:
import datetime as dt
import json
import os
import pathlib

import pandas as pd
import snowflake.connector
from dotenv import find_dotenv, load_dotenv
from googleapiclient.discovery import build
from snowflake.connector.pandas_tools import write_pandas

from youtuber_analysis.assets.raw_youtube import load_channel_config

load_dotenv(find_dotenv(usecwd=True), override=True)

True

In [29]:
account=os.environ["SNOWFLAKE_ACCOUNT"]
user=os.environ["SNOWFLAKE_USER"]
private_key_file=str(pathlib.Path(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]).expanduser())
role=os.environ["SNOWFLAKE_ROLE"]
warehouse=os.environ["SNOWFLAKE_WAREHOUSE"]
database=os.environ["SNOWFLAKE_DATABASE"]
schema="RAW"

In [30]:
print(f"account={account}")
print(f"user={user}")
print(f"private_key_file={private_key_file}")
print(f"role={role}")
print(f"warehouse={warehouse}")
print(f"database={database}")
print(f"schema={schema}")

account=VGBHGMV-NP14382
user=DAGSTER_SVC
private_key_file=/Users/robertmorsch/.snowflake/keys/youtuber_analysis_svc.p8
role=YT_LOADER
warehouse=YT_WH
database=YOUTUBE_ANALYTICS
schema=RAW


## Config

In [31]:
TARGET_TABLE = "YOUTUBE_VIDEOS"
LOAD_TABLE = "YOUTUBE_VIDEOS_LOAD"  # temporary, session-scoped
SNAPSHOT_DATE = dt.date.today().isoformat()

# Set to a list of channel_ids to load a subset, e.g. ["UCnQC_G5Xsjhp9fEJKuIcrSw"].
ONLY_CHANNEL_IDS = None

channels = [
    c
    for c in load_channel_config()
    if not c["channel_id"].startswith("UC_PLACEHOLDER")
    and (ONLY_CHANNEL_IDS is None or c["channel_id"] in ONLY_CHANNEL_IDS)
]
skipped = [c["channel_name"] for c in load_channel_config() if c["channel_id"].startswith("UC_PLACEHOLDER")]

print(f"Snapshot date: {SNAPSHOT_DATE}")
print(f"Loading {len(channels)} channel(s): {[c['channel_name'] for c in channels]}")
if skipped:
    print(f"Skipping placeholder channel(s): {skipped}")

Snapshot date: 2026-09-14
Loading 1 channel(s): ['Ben Shapiro']
Skipping placeholder channel(s): ['Placeholder Channel 2', 'Placeholder Channel 3']


## YouTube API

In [32]:
youtube = build("youtube", "v3", developerKey=os.environ["YOUTUBE_API_KEY"], cache_discovery=False)


def get_channel(channel_id: str) -> dict:
    response = youtube.channels().list(part="snippet,statistics,contentDetails", id=channel_id).execute()
    items = response.get("items", [])
    if not items:
        raise ValueError(f"No channel found for id={channel_id!r}")
    return items[0]


def get_video_ids(playlist_id: str) -> list[str]:
    video_ids = []
    page_token = None
    while True:
        response = (
            youtube.playlistItems()
            .list(part="contentDetails", playlistId=playlist_id, maxResults=50, pageToken=page_token)
            .execute()
        )
        video_ids.extend(item["contentDetails"]["videoId"] for item in response.get("items", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            return video_ids


def get_videos(video_ids: list[str]) -> list[dict]:
    # videos.list costs 1 unit per call no matter how many parts are requested,
    # so pull everything useful while we're here.
    videos = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i : i + 50]
        response = (
            youtube.videos()
            .list(part="snippet,statistics,contentDetails,status,topicDetails", id=",".join(batch))
            .execute()
        )
        videos.extend(response.get("items", []))
    return videos

## Flatten to rows

Everything lands as strings in the load table and is cast to real types in
SQL on insert, which sidesteps pandas/parquet type inference (timezones,
nullable ints). The full API response is kept in `RAW` (a `VARIANT`) so new
fields can be pulled out later without re-hitting the API.

Missing counts stay `NULL` rather than `0`: a creator hiding likes is not the
same as a video having zero likes.

In [33]:
def video_to_row(video: dict, channel: dict, loaded_at: str) -> dict:
    snippet = video.get("snippet", {})
    stats = video.get("statistics", {})
    details = video.get("contentDetails", {})
    return {
        "SNAPSHOT_DATE": SNAPSHOT_DATE,
        "LOADED_AT": loaded_at,
        "CHANNEL_ID": channel["channel_id"],
        "CHANNEL_NAME": channel["channel_name"],
        "CHANNEL_PHASE": str(channel["phase"]),
        "VIDEO_ID": video["id"],
        "TITLE": snippet.get("title"),
        "DESCRIPTION": snippet.get("description"),
        "PUBLISHED_AT": snippet.get("publishedAt"),
        "DURATION": details.get("duration"),
        "CATEGORY_ID": snippet.get("categoryId"),
        "DEFAULT_AUDIO_LANGUAGE": snippet.get("defaultAudioLanguage"),
        "LIVE_BROADCAST_CONTENT": snippet.get("liveBroadcastContent"),
        "TAGS": json.dumps(snippet.get("tags", [])),
        "TOPIC_CATEGORIES": json.dumps(video.get("topicDetails", {}).get("topicCategories", [])),
        "VIEW_COUNT": stats.get("viewCount"),
        "LIKE_COUNT": stats.get("likeCount"),
        "COMMENT_COUNT": stats.get("commentCount"),
        "RAW": json.dumps(video),
    }

## Snowflake

In [34]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    private_key_file=str(pathlib.Path(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]).expanduser()),
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="RAW",
)
conn.cursor().execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA()").fetchone()

('DAGSTER_SVC', 'YT_LOADER', 'YOUTUBE_ANALYTICS', 'RAW')

In [35]:
conn.cursor().execute(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        SNAPSHOT_DATE           DATE          NOT NULL,
        LOADED_AT               TIMESTAMP_TZ  NOT NULL,
        CHANNEL_ID              VARCHAR       NOT NULL,
        CHANNEL_NAME            VARCHAR,
        CHANNEL_PHASE           NUMBER(1),
        VIDEO_ID                VARCHAR       NOT NULL,
        TITLE                   VARCHAR,
        DESCRIPTION             VARCHAR,
        PUBLISHED_AT            TIMESTAMP_TZ,
        DURATION                VARCHAR,      -- ISO 8601, e.g. PT1H2M3S
        CATEGORY_ID             VARCHAR,
        DEFAULT_AUDIO_LANGUAGE  VARCHAR,
        LIVE_BROADCAST_CONTENT  VARCHAR,
        TAGS                    ARRAY,
        TOPIC_CATEGORIES        ARRAY,
        VIEW_COUNT              NUMBER,
        LIKE_COUNT              NUMBER,
        COMMENT_COUNT           NUMBER,
        RAW                     VARIANT
    )
    COMMENT = 'All YouTube videos across configured channels, one row per video per snapshot_date'
""")

In [36]:
def load_rows(rows: list[dict], channel_id: str) -> int:
    """Stage rows in a temp table, then swap them in for this snapshot_date
    and channel in one transaction."""
    df = pd.DataFrame(rows)
    write_pandas(
        conn,
        df,
        LOAD_TABLE,
        auto_create_table=True,
        table_type="temporary",
        overwrite=True,
    )

    cur = conn.cursor()
    try:
        cur.execute("BEGIN")
        cur.execute(
            f"DELETE FROM {TARGET_TABLE} WHERE SNAPSHOT_DATE = %s AND CHANNEL_ID = %s",
            (SNAPSHOT_DATE, channel_id),
        )
        cur.execute(f"""
            INSERT INTO {TARGET_TABLE}
            SELECT
                "SNAPSHOT_DATE"::DATE,
                "LOADED_AT"::TIMESTAMP_TZ,
                "CHANNEL_ID",
                "CHANNEL_NAME",
                "CHANNEL_PHASE"::NUMBER(1),
                "VIDEO_ID",
                "TITLE",
                "DESCRIPTION",
                TRY_TO_TIMESTAMP_TZ("PUBLISHED_AT"),
                "DURATION",
                "CATEGORY_ID",
                "DEFAULT_AUDIO_LANGUAGE",
                "LIVE_BROADCAST_CONTENT",
                PARSE_JSON("TAGS")::ARRAY,
                PARSE_JSON("TOPIC_CATEGORIES")::ARRAY,
                TRY_TO_NUMBER("VIEW_COUNT"),
                TRY_TO_NUMBER("LIKE_COUNT"),
                TRY_TO_NUMBER("COMMENT_COUNT"),
                PARSE_JSON("RAW")
            FROM {LOAD_TABLE}
        """)
        inserted = cur.rowcount
        cur.execute("COMMIT")
    except Exception:
        cur.execute("ROLLBACK")
        raise
    return inserted

## Run

Loads one channel at a time so a failure (bad channel ID, quota exhausted)
doesn't lose the channels that already succeeded.

In [37]:
results = []
for channel in channels:
    name = channel["channel_name"]
    try:
        raw_channel = get_channel(channel["channel_id"])
        uploads_playlist_id = raw_channel["contentDetails"]["relatedPlaylists"]["uploads"]
        video_ids = get_video_ids(uploads_playlist_id)
        videos = get_videos(video_ids)

        loaded_at = dt.datetime.now(dt.timezone.utc).isoformat()
        rows = [video_to_row(v, channel, loaded_at) for v in videos]
        inserted = load_rows(rows, channel["channel_id"]) if rows else 0

        # 1 (channels) + one playlistItems page and one videos call per 50 videos
        units = 1 + 2 * -(-len(video_ids) // 50)
        results.append({"channel": name, "playlist_videos": len(video_ids), "inserted": inserted, "quota_units": units})
        print(f"{name}: {len(video_ids)} in playlist, {inserted} rows inserted (~{units} quota units)")
    except Exception as e:
        results.append({"channel": name, "error": repr(e)})
        print(f"{name}: FAILED - {e!r}")

pd.DataFrame(results)

Ben Shapiro: 10802 in playlist, 10802 rows inserted (~435 quota units)


,channel,playlist_videos,inserted,quota_units
0,Ben Shapiro,10802,10802,435


`playlist_videos` can exceed `inserted`: the uploads playlist still lists
videos that have since gone private or been deleted, and `videos.list` returns
nothing for them.

## Check

In [38]:
conn.cursor().execute(f"""
    SELECT SNAPSHOT_DATE, CHANNEL_NAME, COUNT(*) AS VIDEOS, MIN(PUBLISHED_AT) AS FIRST_VIDEO, MAX(PUBLISHED_AT) AS LATEST_VIDEO, SUM(VIEW_COUNT) AS TOTAL_VIEWS
    FROM {TARGET_TABLE}
    GROUP BY 1, 2
    ORDER BY 1 DESC, 2
""").fetch_pandas_all()

,SNAPSHOT_DATE,CHANNEL_NAME,VIDEOS,FIRST_VIDEO,LATEST_VIDEO,TOTAL_VIEWS
0,2026-09-14,Ben Shapiro,10802,2016-11-06 21:49:26-08:00,2026-09-14 16:01:51-07:00,4793785773


In [39]:
conn.close()